# 97 — GNN Bucket Action Selector

Heterogeneous GNN (tripartite: planet ↔ action ↔ planet) that selects attacks and
ship counts, trained by imitating the 90-Simulate heuristic.
Output: [[src_id, dst_id, eta, ships_to_send], ...]

In [ ]:
%run 96-library.py
%run 90-Simulate10Next_Conqueror2_Supplier_prod_per_step.py
%run 97-library.py
import torch
import torch.nn as nn
from torch_geometric.nn import SAGEConv
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML, display
import math, copy

In [ ]:
_COLORS = {0: 'steelblue', 1: 'tomato', -1: '#888888'}

def make_animation(snapshots, title='', interval=150):
    fig, ax = plt.subplots(figsize=(6, 6))
    fig.patch.set_facecolor('#111122')
    def draw(frame):
        snap = snapshots[frame]
        ax.cla()
        ax.set_xlim(0, 100); ax.set_ylim(100, 0)
        ax.set_aspect('equal'); ax.set_facecolor('#111122')
        ax.tick_params(colors='#aaaaaa')
        for sp in ax.spines.values(): sp.set_edgecolor('#444444')
        ax.set_title(f"{title}  (step {snap['step']})", color='white', fontsize=11)
        ax.add_patch(plt.Circle((50, 50), 10, color='gold', zorder=2, alpha=0.9))
        for p in snap['planets']:
            pid, owner, x, y, radius, ships, production = p
            c = _COLORS.get(owner, '#888888')
            ax.add_patch(plt.Circle((x, y), radius, color=c, alpha=0.85, zorder=3))
            ax.text(x, y,   str(ships),         ha='center', va='center', color='white', fontsize=7, fontweight='bold', zorder=4)
            ax.text(x, y+2, str(pid),            ha='center', va='center', color='red',   fontsize=7, fontweight='bold', zorder=4)
            ax.text(x, y-2, '+'+str(production), ha='center', va='center', color='white', fontsize=5, fontweight='bold', zorder=4)
        return []
    ani = animation.FuncAnimation(fig, draw, frames=len(snapshots), interval=interval)
    plt.close()
    return HTML(ani.to_jshtml())

In [ ]:
data0, snaps0, ai0 = generate_sample_97(42)
n_act = data0['action'].x.shape[0]
n_pos = int(data0['action'].y.sum().item())
print(f"Planets: {data0['planet'].x.shape[0]}")
print(f"Action nodes: {n_act}  |  Positive (heuristic selected): {n_pos}")
print(f"Spawns edges: {data0['planet','spawns','action'].edge_index.shape[1]}")
print(f"Attacks edges: {data0['action','attacks','planet'].edge_index.shape[1]}")
make_animation(snaps0, title='Sample seed=42', interval=200)

In [ ]:
print("Generating train dataset (1000 samples)...")
train_dataset = [generate_sample_97(i) for i in range(1000)]
print("Generating test dataset (20 samples)...")
test_dataset  = [generate_sample_97(10000 + i) for i in range(20)]

train_pos = sum(int(d.y.sum()) for d, _, _ in train_dataset)
train_tot = sum(d['action'].x.shape[0] for d, _, _ in train_dataset)
print(f"Train: {train_tot} action nodes total, {train_pos} positive ({100*train_pos/max(train_tot,1):.1f}%)")

In [ ]:
import os

class GNNActionSelector(nn.Module):
    """Tripartite GNN: 3 SAGEConv passes, 2 output heads per action node.

    Layer 1: planet -> action  (source context into action)
             planet -> master, master -> planet
    Layer 2: action -> planet  (action context into destination)
    Layer 3: planet -> action  (enriched destination context back into action)

    Output heads per action node:
      select_head: Linear(H->1) raw logit  (BCEWithLogitsLoss)
      ships_head:  Linear(H->1)+Sigmoid    (MSELoss vs normalised ships target)
    """
    def __init__(self, hidden_dim: int = 64):
        super().__init__()
        H = hidden_dim
        self.master_lin = nn.Linear(6,  H)
        self.planet_lin = nn.Linear(22, H)
        self.action_lin = nn.Linear(2,  H)

        # Layer 1: planet->action, planet->master, master->planet
        self.sage1_pa = SAGEConv(H, H)
        self.sage1_pm = SAGEConv(H, H)
        self.sage1_mp = SAGEConv(H, H)

        # Layer 2: action->planet
        self.sage2_ap = SAGEConv(H, H)

        # Layer 3: planet->action
        self.sage3_pa = SAGEConv(H, H)

        self.select_head = nn.Linear(H, 1)
        self.ships_head  = nn.Sequential(nn.Linear(H, 1), nn.Sigmoid())

    def forward(self, data):
        h_m = torch.relu(self.master_lin(data['master'].x))
        h_p = torch.relu(self.planet_lin(data['planet'].x))
        h_a = torch.relu(self.action_lin(data['action'].x))

        ei_spawns  = data['planet', 'spawns',    'action'].edge_index
        ei_attacks = data['action', 'attacks',   'planet'].edge_index
        ei_pm      = data['planet', 'to_master', 'master'].edge_index
        ei_mp      = data['master', 'to_planet', 'planet'].edge_index

        # Layer 1
        h_a = torch.relu(self.sage1_pa((h_p, h_a), ei_spawns))
        h_m = torch.relu(self.sage1_pm((h_p, h_m), ei_pm))
        h_p = torch.relu(self.sage1_mp((h_m, h_p), ei_mp))

        # Layer 2
        h_p = torch.relu(self.sage2_ap((h_a, h_p), ei_attacks))

        # Layer 3
        h_a = torch.relu(self.sage3_pa((h_p, h_a), ei_spawns))

        return self.select_head(h_a).squeeze(-1), self.ships_head(h_a).squeeze(-1)


MODEL_PATH = "97-model.pt"
model = GNNActionSelector(hidden_dim=64)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")

# Shape check
data0, _, _ = train_dataset[0]
if data0['action'].x.shape[0] > 0:
    sl, sh = model(data0)
    n_act = data0['action'].x.shape[0]
    assert sl.shape == (n_act,), f"select logit shape {sl.shape}"
    assert sh.shape == (n_act,), f"ships head shape {sh.shape}"
    print(f"Forward pass OK — {n_act} action nodes")

if os.path.exists(MODEL_PATH):
    model.load_state_dict(torch.load(MODEL_PATH, weights_only=True))
    model.eval()
    _model_loaded = True
    print(f"Loaded model from {MODEL_PATH}")
else:
    _model_loaded = False
    print("No saved model — will train from scratch.")

In [ ]:
if _model_loaded:
    print("Model already loaded — skipping training.")
else:
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    bce = nn.BCEWithLogitsLoss()
    mse = nn.MSELoss()

    model.train()
    for epoch in range(50):
        total_loss = total_sel = total_shp = 0.0
        n_graphs = 0
        for data, _, _ in train_dataset:
            if data['action'].x.shape[0] == 0:
                continue
            optimizer.zero_grad()
            sel_logit, ships_pred = model(data)
            loss_sel = bce(sel_logit, data['action'].y)
            loss_shp = mse(ships_pred, data['action'].ships_target)
            loss = loss_sel + loss_shp
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            total_sel  += loss_sel.item()
            total_shp  += loss_shp.item()
            n_graphs   += 1
        if epoch % 10 == 0:
            print(f"Epoch {epoch:3d} | total {total_loss/n_graphs:.4f} | bce {total_sel/n_graphs:.4f} | mse {total_shp/n_graphs:.4f}")

    torch.save(model.state_dict(), MODEL_PATH)
    print(f"Training complete. Saved to {MODEL_PATH}.")